## Day 17 Completion Note

- The common CIC-IDS2018 test input was recreated successfully.
- Test input contained 209,715 rows and 78 numeric features.
- Invalid numeric values were handled using training-data-derived medians.
- All five saved baseline models were loaded successfully:
  - Decision Tree
  - Random Forest
  - AdaBoost
  - XGBoost
  - LightGBM
- Predictions were generated for all five models.
- Each model produced 209,715 predictions.
- No missing predictions were found.
- XGBoost and LightGBM encoded predictions were converted back to the original class labels.
- The predictions were saved to:
  `results/day17_all_model_predictions.csv`
- Saved prediction file was successfully verified.

### Status
Day 17 – Generate Predictions: **COMPLETED**

In [8]:
import os
import pandas as pd

print("Prediction file exists:", os.path.exists(predictions_path))

loaded_predictions = pd.read_csv(predictions_path)

print("\nShape:", loaded_predictions.shape)

print("\nColumns:")
print(loaded_predictions.columns.tolist())

print("\nFirst 5 rows:")
print(loaded_predictions.head())

Prediction file exists: True

Shape: (209715, 6)

Columns:
['Actual_Label', 'Decision_Tree', 'Random_Forest', 'AdaBoost', 'XGBoost', 'LightGBM']

First 5 rows:
     Actual_Label   Decision_Tree   Random_Forest        AdaBoost  \
0          Benign          Benign          Benign          Benign   
1          Benign          Benign          Benign          Benign   
2          Benign          Benign          Benign          Benign   
3          Benign          Benign          Benign          Benign   
4  FTP-BruteForce  FTP-BruteForce  FTP-BruteForce  FTP-BruteForce   

          XGBoost        LightGBM  
0          Benign          Benign  
1          Benign          Benign  
2          Benign          Benign  
3          Benign          Benign  
4  FTP-BruteForce  FTP-BruteForce  


In [7]:
predictions_df = pd.DataFrame({
    "Actual_Label": y_test.to_numpy(),
    "Decision_Tree": predictions["Decision Tree"],
    "Random_Forest": predictions["Random Forest"],
    "AdaBoost": predictions["AdaBoost"],
    "XGBoost": predictions["XGBoost"],
    "LightGBM": predictions["LightGBM"]
})

predictions_path = "../results/day17_all_model_predictions.csv"

predictions_df.to_csv(
    predictions_path,
    index=False
)

print("Predictions saved successfully.")
print("File path:", predictions_path)
print("Rows:", len(predictions_df))
print("Columns:", predictions_df.shape[1])

Predictions saved successfully.
File path: ../results/day17_all_model_predictions.csv
Rows: 209715
Columns: 6


In [6]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit using the original target labels
label_encoder.fit(y_train)

predictions["XGBoost"] = label_encoder.inverse_transform(
    predictions["XGBoost"].astype(int)
)

predictions["LightGBM"] = label_encoder.inverse_transform(
    predictions["LightGBM"].astype(int)
)

print("XGBoost labels:", np.unique(predictions["XGBoost"]))
print("LightGBM labels:", np.unique(predictions["LightGBM"]))

XGBoost labels: ['Benign' 'FTP-BruteForce' 'SSH-Bruteforce']
LightGBM labels: ['Benign' 'FTP-BruteForce' 'SSH-Bruteforce']


In [5]:
for name, pred in predictions.items():
    print(f"\n{name}")
    print("Prediction count:", len(pred))
    print("Unique labels:", np.unique(pred))
    print("Missing predictions:", pd.isna(pred).sum())


Decision Tree
Prediction count: 209715
Unique labels: ['Benign' 'FTP-BruteForce' 'SSH-Bruteforce']
Missing predictions: 0

Random Forest
Prediction count: 209715
Unique labels: ['Benign' 'FTP-BruteForce' 'SSH-Bruteforce']
Missing predictions: 0

AdaBoost
Prediction count: 209715
Unique labels: ['Benign' 'FTP-BruteForce' 'SSH-Bruteforce']
Missing predictions: 0

XGBoost
Prediction count: 209715
Unique labels: [0 1 2]
Missing predictions: 0

LightGBM
Prediction count: 209715
Unique labels: [0 1 2]
Missing predictions: 0


In [4]:
predictions = {}

for name, model in loaded_models.items():
    print(f"Generating predictions: {name}")

    predictions[name] = model.predict(X_test_clean)

    print(
        f"{name}: {len(predictions[name])} predictions generated."
    )

print("\nAll 5 prediction sets generated successfully.")

Generating predictions: Decision Tree
Decision Tree: 209715 predictions generated.
Generating predictions: Random Forest
Random Forest: 209715 predictions generated.
Generating predictions: AdaBoost
AdaBoost: 209715 predictions generated.
Generating predictions: XGBoost
XGBoost: 209715 predictions generated.
Generating predictions: LightGBM
LightGBM: 209715 predictions generated.

All 5 prediction sets generated successfully.


In [3]:
import joblib

model_paths = {
    "Decision Tree": "../models/decision_tree_baseline.joblib",
    "Random Forest": "../models/random_forest_baseline.joblib",
    "AdaBoost": "../models/adaboost_baseline.joblib",
    "XGBoost": "../models/xgboost_baseline.joblib",
    "LightGBM": "../models/lightgbm_baseline.joblib"
}

loaded_models = {}

for name, path in model_paths.items():
    loaded_models[name] = joblib.load(path)
    print(f"{name} loaded successfully.")

print("\nAll 5 baseline models loaded successfully.")

Decision Tree loaded successfully.
Random Forest loaded successfully.
AdaBoost loaded successfully.
XGBoost loaded successfully.
LightGBM loaded successfully.

All 5 baseline models loaded successfully.


In [2]:
X_test_clean = X_test.copy()

# Convert infinite values to NaN
X_test_clean = X_test_clean.replace([np.inf, -np.inf], np.nan)

# Calculate training medians
X_train_clean = X_train.copy()
X_train_clean = X_train_clean.replace([np.inf, -np.inf], np.nan)
train_medians = X_train_clean.median()

# Fill test missing values using training medians only
X_test_clean = X_test_clean.fillna(train_medians)

# Verify
print("Test NaN:", X_test_clean.isna().sum().sum())
print("Test Inf:", np.isinf(X_test_clean).sum().sum())
print(
    "Testing input is finite:",
    np.isfinite(X_test_clean.to_numpy()).all()
)

Test NaN: 0
Test Inf: 0
Testing input is finite: True


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../cic.csv")

X = df.drop(columns=["Label"])
y = df["Label"]

X_numeric = X.select_dtypes(include="number")

X_train, X_test, y_train, y_test = train_test_split(
    X_numeric,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Test features:", X_test.shape)
print("Test target  :", y_test.shape)

Test features: (209715, 78)
Test target  : (209715,)


# Day 17 – Generate Predictions

## Objective
Generate and save test-set predictions from all five baseline ML models.

## Models
1. Decision Tree
2. Random Forest
3. AdaBoost
4. XGBoost
5. LightGBM

## Prediction Input
- Dataset: CIC-IDS2018
- Test rows: 209,715
- Numeric features: 78
- Same 80/20 stratified split
- `random_state=42`
- Same training-data-derived median cleaning

## Day 17 Scope
- Load the common test input.
- Load the five saved baseline models.
- Generate predictions for the test set.
- Verify prediction count and labels.
- Save predictions for later analysis.

No hyperparameter tuning will be performed on Day 17.